In [71]:
from e3nn import o3
import torch

In [4]:
irrep = o3.Irreps('4x0e + 4x1o + 4x2e')

irrep

irrep.slices()

[slice(0, 4, None), slice(4, 16, None), slice(16, 36, None)]

In [5]:
from allegro.nn._strided._layout import StridedLayout

In [6]:
strided_irrep = StridedLayout(irrep, pad_to_multiple= 1)

In [7]:
# Seems like a major format. You can represent inexed in the form n m l
strided_irrep.indexes_to_strided.reshape((4, 9))

tensor([[ 0,  4,  5,  6, 16, 17, 18, 19, 20],
        [ 1,  7,  8,  9, 21, 22, 23, 24, 25],
        [ 2, 10, 11, 12, 26, 27, 28, 29, 30],
        [ 3, 13, 14, 15, 31, 32, 33, 34, 35]])

In [8]:
# transform back
strided_irrep.indexes_to_catted

tensor([ 0,  9, 18, 27,  1,  2,  3, 10, 11, 12, 19, 20, 21, 28, 29, 30,  4,  5,
         6,  7,  8, 13, 14, 15, 16, 17, 22, 23, 24, 25, 26, 31, 32, 33, 34, 35])

In [9]:
# this one is fine but do not set high spherical harmonics in right order
import numpy as np
np.array(range(36)).reshape((9, 4))

array([[ 0,  1,  2,  3],
       [ 4,  5,  6,  7],
       [ 8,  9, 10, 11],
       [12, 13, 14, 15],
       [16, 17, 18, 19],
       [20, 21, 22, 23],
       [24, 25, 26, 27],
       [28, 29, 30, 31],
       [32, 33, 34, 35]])

In [37]:
instr = []
# Instructions
irreps_in1 = o3.Irreps('4x0e + 4x1o + 4x2e')
irreps_in2 = o3.Irreps('4x0e + 4x1o + 4x2e')
irreps_out = o3.Irreps('4x0e + 4x1o + 4x2e')


tmp_i_out: int = 0
for i_out, (_, ir_out) in enumerate(irreps_out):
    for i_1, (_, ir_in1) in enumerate(irreps_in1):
        for i_2, (_, ir_in2) in enumerate(irreps_in2):
            if ir_out in ir_in1 * ir_in2:
                instr.append((i_1, i_2, i_out))
                
                tmp_i_out += 1

In [38]:
instr

[(0, 0, 0),
 (1, 1, 0),
 (2, 2, 0),
 (0, 1, 1),
 (1, 0, 1),
 (1, 2, 1),
 (2, 1, 1),
 (0, 2, 2),
 (1, 1, 2),
 (2, 0, 2),
 (2, 2, 2)]

In [39]:
irreps_out[2]

4x2ee

In [40]:
# Example of two functions
#def codegen_strided_tensor_product_forward(
#    irreps_in1: o3.Irreps,
#    in1_var: List[float],
#    irreps_in2: o3.Irreps,
#    in2_var: List[float],
#    irreps_out: o3.Irreps,
#    out_var: List[float],
#    instructions: List[Instruction],
#    normalization: str = "component",
#    shared_weights: bool = False,
#    specialized_code: bool = True,
#    sparse_mode: Optional[str] = None,
#    pad_to_alignment: int = 1,
#) -> Optional[fx.GraphModule]:

#def Contracter(
#    irreps_in1,
#    irreps_in2,
#    irreps_out,
#    instructions: List[Tuple[int, int, int]],
#    has_weight: bool,
#    connection_mode: str,
#    pad_to_alignment: int = 1,
#    shared_weights: bool = False,
#    sparse_mode: Optional[str] = None,
#):

In [129]:
from e3nn.o3 import Instruction

layer_idx = 1

connection_mode=(
                    "uuu" if layer_idx > 0 or self.embed_initial_edge else "uvv"
                )
has_weight = True

instructions=[
            Instruction(
                i_in1,
                i_in2,
                i_out,
                connection_mode,
                has_weight,
                1.0,
                {
                    "uvw": (
                        irreps_in1[i_in1].mul,
                        irreps_in2[i_in2].mul,
                        irreps_out[i_out].mul,
                    ),
                    "uvu": (irreps_in1[i_in1].mul, irreps_in2[i_in2].mul),
                    "uvv": (irreps_in1[i_in1].mul, irreps_in2[i_in2].mul),
                    "uuw": (irreps_in1[i_in1].mul, irreps_out[i_out].mul),
                    "uuu": (irreps_in1[i_in1].mul,),
                    "uvuv": (
                        irreps_in1[i_in1].mul,
                        irreps_in2[i_in2].mul,
                    ),
                }[connection_mode],
            )
            for i_in1, i_in2, i_out in instr
        ]

In [130]:
# Gen big w3j

layout_in1 = StridedLayout(irreps_in1)
layout_in2 = StridedLayout(irreps_in2)
layout_out = StridedLayout(irreps_out)

w3j_values = []
w3j_index = []


for ins_i, ins in enumerate(instructions):
    mul_ir_in1 = layout_in1.base_irreps[ins.i_in1]
    mul_ir_in2 = layout_in2.base_irreps[ins.i_in2]
    mul_ir_out = layout_out.base_irreps[ins.i_out]
    
    
    this_w3j = o3.wigner_3j(mul_ir_in1.ir.l, mul_ir_in2.ir.l, mul_ir_out.ir.l)
    this_w3j_index = this_w3j.nonzero()
    w3j_values.append(
        this_w3j[this_w3j_index[:, 0], this_w3j_index[:, 1], this_w3j_index[:, 2]]
    )
    
    
    this_w3j_index[:, 0] += layout_in1.base_irreps[: ins.i_in1].dim
    this_w3j_index[:, 1] += layout_in2.base_irreps[: ins.i_in2].dim
    this_w3j_index[:, 2] += layout_out.base_irreps[: ins.i_out].dim    
    
    #print(mul_ir_out)
    
    
    # Now need to flatten the index to be for [pk][ij]
    w3j_index.append(
        torch.cat(
            (
                (ins_i if ins.has_weight else 0)  # unweighted all go in first path
                * layout_out.base_dim
                + this_w3j_index[:, 2].unsqueeze(-1),
                this_w3j_index[:, 0].unsqueeze(-1) * layout_in2.base_dim
                + this_w3j_index[:, 1].unsqueeze(-1),
            ),
            dim=1,
        )
    )

num_paths: int = len(instructions) if has_weight else 1
    
w3j = torch.sparse_coo_tensor(
        indices=torch.cat(w3j_index, dim=0).t(),
        values=torch.cat(w3j_values, dim=0),
        size=(
            num_paths * layout_out.base_dim,
            layout_in1.base_dim * layout_in2.base_dim,
        ),
    ).coalesce()

In [131]:
w3j_i_indexes = torch.div(
    w3j.indices()[1], layout_in1.base_dim, rounding_mode="floor"
)
w3j_j_indexes = w3j.indices()[1] % layout_in1.base_dim
w3j_is_ij_diagonal = (layout_in1.base_dim == layout_in2.base_dim) and torch.all(
    w3j_i_indexes == w3j_j_indexes
)

In [132]:
w3j_i_indexes

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 0, 0, 0, 1, 2, 3, 1, 1, 2, 3, 1, 2, 3, 1, 2,
        3, 3, 4, 5, 6, 8, 5, 6, 7, 4, 6, 7, 8, 0, 0, 0, 0, 0, 1, 3, 1, 2, 1, 2,
        3, 2, 3, 1, 3, 4, 5, 6, 7, 8, 4, 5, 6, 7, 4, 5, 5, 6, 7, 8, 4, 5, 6, 7,
        8, 4, 5, 6, 7, 7, 8, 5, 6, 7, 8])

In [133]:
w3j_j_indexes

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 1, 2, 3, 0, 0, 0, 6, 8, 5, 4, 5, 6, 7, 4, 7,
        6, 8, 3, 2, 1, 1, 1, 2, 3, 1, 3, 2, 3, 4, 5, 6, 7, 8, 3, 1, 2, 1, 1, 2,
        3, 3, 2, 1, 3, 0, 0, 0, 0, 0, 6, 7, 4, 5, 7, 6, 8, 5, 4, 5, 4, 5, 6, 7,
        8, 5, 4, 7, 6, 8, 7, 5, 8, 7, 6])

In [134]:
w3j_is_ij_diagonal

tensor(False)

In [135]:
layout_in1.base_dim

9

In [139]:
kij_shape = (
                layout_out.base_dim,
                layout_in1.base_dim,
                layout_in2.base_dim,
            )
w3j = (
    w3j.to_dense()
    .reshape(((num_paths,) if num_paths > 1 else tuple()) + kij_shape)
.contiguous()
)

In [141]:
w3j.shape

torch.Size([11, 9, 9, 9])